In [ ]:
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import levene, ks_2samp

import cafo_iowa.db.session as s
import cafo_iowa.db.models as m

from cafo_iowa.utils.visualize import simple_map
from cafo_iowa.data.helpers.geo import merge_overlapping_geometries

In [ ]:
session = s.get_session()
engine = session.get_bind()

In [ ]:
# get all labelled tiles
naip_qt_ids = session.query(m.CFAnnotations.naip_qt_id).all()
batches = session.query(m.CFAnnotations.batch_name).all()
naip_qt_ids = [x[0] for x in naip_qt_ids]
naip_qt_ids_str = "','".join(naip_qt_ids)  # Convert list to a string for SQL IN clause

In [ ]:
barns = gpd.read_postgis(
    f"""
    SELECT 
        b.id AS barn_id, 
        bp.parcel_id,
        b.geometry
    FROM 
        processed.{m.Barns.__tablename__} b
        LEFT JOIN
        processed.{m.BarnsParcels.__tablename__} bp
    ON
        b.id = bp.barn_id
    """,
    engine,
    geom_col="geometry",
)

In [ ]:
# permits
permits = gpd.read_postgis(
    f"""
    SELECT 
        p.id AS permit_id,
        p.facilityna,
        p.address,
        p.city,
        p.state, 
        p.zip,
        p.parcel_id, 
        animal_units, 
        swine_animal_units, 
        swine_gest, 
        swine_gilt,
        swine_grow, 
        swine_nurs,
        swine_sow, 
        swine_wean,
        animal_type, 
        swine_type,
        p.geometry
    FROM 
        processed.{m.Permits.__tablename__} p 
    WHERE
        p.naip_qt_id IN ('{naip_qt_ids_str}');
    """,
    engine,
    geom_col="geometry",
)

In [ ]:
# Parcels
parcel_ids = permits.parcel_id.unique()
parcel_ids = parcel_ids[~pd.isna(parcel_ids)]
parcel_ids = "','".join(parcel_ids)

parcels = gpd.read_postgis(
    f"""
    SELECT 
        id AS parcel_id, 
        geometry 
    FROM 
        processed.{m.Parcels.__tablename__} p
    WHERE 
        p.id IN ('{parcel_ids}');
    """,
    engine,
    geom_col="geometry",
)

In [ ]:
# Tiles
naip_qt = gpd.read_postgis(
    f"""
    SELECT
        id AS naip_qt_id,
        geometry,
        geometry_buffer
    FROM
        processed.{m.Naip21QT.__tablename__}
    WHERE
        id IN ('{naip_qt_ids_str}');
    """,
    engine,
    geom_col="geometry",
)

In [ ]:
print("Batches labelled: ", sorted(batches))
print("Naip tiles labelled: ", naip_qt.naip_qt_id.nunique())
print("Annotations: ", barns.shape[0])
print("Permits in labelled tiles: ", permits.shape[0])
print("Parcels associated with permits: ", parcels.shape[0])
print(
    "Mean number of annotations per parcel: ",
    barns.groupby("parcel_id").size().mean(),
)
print(
    "Parcels with 0 barns: ",
    parcels[~parcels.parcel_id.isin(barns.parcel_id)].shape[0],
)

In [ ]:
# save as geojsons for manual inspection
barns.to_file("data/manual_review/barns.geojson", driver="GeoJSON")
naip_qt.to_file("data/manual_review/naip_qt.geojson", driver="GeoJSON")
permits.to_file("data/manual_review/permits.geojson", driver="GeoJSON")
parcels.to_file("data/manual_review/parcels.geojson", driver="GeoJSON")

In [ ]:
# annotations where parcel_id is in parcels
barns_within_parcels = barns[barns.parcel_id.isin(parcels.parcel_id)]
barns_outside_parcels = barns[~barns.parcel_id.isin(parcels.parcel_id)]

In [ ]:
# Describe the data
barns_within_parcels = barns_within_parcels.to_crs(epsg=3857)
barns_outside_parcels = barns_outside_parcels.to_crs(epsg=3857)
barns_within_parcels["barn_area"] = barns_within_parcels.area
barns_outside_parcels["barn_area"] = barns_outside_parcels.area
barns_within_parcels = barns_within_parcels.to_crs(parcels.crs)
barns_outside_parcels = barns_outside_parcels.to_crs(parcels.crs)

print("Barns within parcels: ", len(barns_within_parcels))
print("Barns outside parcels: ", len(barns_outside_parcels))
print("Mean barn area within parcels: ", barns_within_parcels.barn_area.mean())
print(
    "Mean barn area outside parcels: ",
    barns_outside_parcels.barn_area.mean(),
)

In [ ]:
map = simple_map(
    "EPSG:4326",
    barns_within_parcels,
    parcels,
    barns_outside_parcels,
    naip_qt,
    permits,
)
# save html of map
map.save("output/plots/map.html")

In [ ]:
parcels2 = gpd.sjoin(parcels, permits, how="left", op="intersects")
parcels2 = parcels2.drop(columns=["index_right", "parcel_id_right"])
parcels2 = parcels2.rename(columns={"parcel_id_left": "parcel_id"})

barns = barns.to_crs(epsg=3857)
barns["barn_area"] = barns.geometry.area
barns = barns.to_crs(parcels.crs)

# add barn area to parcels2
parcels2 = parcels2.merge(
    barns.groupby("parcel_id")["barn_area"].sum().reset_index(), on="parcel_id"
)
parcels2["animal_type_grp"] = parcels2.animal_type.replace(
    {
        "cattle-swine": "other",
        "chicken-swine": "other",
        "cattle-chicken": "other",
        "cattle-horses": "other",
        "cattle-horses-swine": "other",
        "horses-swine": "other",
        "horses": "other",
        "cattle-chicken-swine": "other",
        "cattle-horses-swine": "other",
    }
)

In [ ]:
# create a set of separate scatter plots of barn size by animal count for each animal type

plt.figure(figsize=(10, 10))
plt.suptitle("Barn Size vs Animal Count by Animal Type")
for i, animal_type in enumerate(parcels2.animal_type_grp.unique()):
    plt.subplot(2, 2, i + 1)
    plt.scatter(
        x=parcels2[parcels2.animal_type == animal_type].animal_units,
        y=parcels2[parcels2.animal_type == animal_type].barn_area,
        alpha=0.3,
        color="blue",
    )
    plt.ylabel("Barn Area (m^2)")
    plt.xlabel("Animal Units")
    # log scale
    plt.yscale("log")
    plt.xscale("log")
    # add line at 1000 animal units
    plt.axvline(1000, color="red", linestyle="--")
    plt.title(animal_type)
plt.tight_layout()

In [ ]:
# Create the animal_type_grp column
categories_to_keep = ["wean", "grow", "nursery"]
parcels2["swine_type_grp"] = parcels2.swine_type
parcels2["swine_type_grp"] = parcels2.swine_type_grp.replace(
    {
        "grow-nursery": "nursery",
        "nursery-wean": "nursery",
    }
)

parcels2["swine_type_grp"] = parcels2.swine_type_grp.apply(
    lambda x: x if x in categories_to_keep else "other"
)

In [ ]:
parcels3 = parcels2[parcels2.animal_type_grp == "swine"]
# create separate plots of barn size by animal count for swine, hue by swine type, by each swine type
plt.figure(figsize=(10, 10))
plt.suptitle("Barn Size vs Animal Count by Swine Type")
for i, swine_type in enumerate(parcels3.swine_type_grp.unique()):
    plt.subplot(2, 2, i + 1)
    sns.scatterplot(
        x="animal_units",
        y="barn_area",
        data=parcels3[parcels3.swine_type_grp == swine_type],
        alpha=0.5,
    )
    # add the correlation line
    x = parcels3[parcels3.swine_type_grp == swine_type].animal_units
    y = parcels3[parcels3.swine_type_grp == swine_type].barn_area
    m, b = np.polyfit(np.log(x), np.log(y), 1)
    plt.plot(x, np.exp(m * np.log(x) + b), color="black")
    # add the correlation coefficient
    r = np.corrcoef(np.log(x), np.log(y))[0, 1]
    plt.text(0.1, 0.9, f"r = {r:.2f}", transform=plt.gca().transAxes)
    plt.ylabel("Barn Area (m^2)")
    plt.xlabel("Animal Units")
    plt.yscale("log")
    plt.xscale("log")
    plt.axvline(500, color="red", linestyle="--")
    plt.axvline(1000, color="red", linestyle="--")
    plt.title(swine_type + " (n = " + str(len(x)) + ")")
    plt.tight_layout()

In [ ]:
# Determine the range of barn_area values across all swine types and animal units
min_barn_area = parcels3.barn_area.min()
max_barn_area = parcels3.barn_area.max()
bin_width = 1000  # Define a specific bin width
bins = np.arange(
    min_barn_area, max_barn_area + bin_width, bin_width
)  # Create bins with the specified width

# Create histograms by swine_type_grp
plt.figure(figsize=(15, 12))
plt.suptitle(
    "Relative Frequency of Barn Size Distribution by Swine Type Group and Animal Units"
)

for i, swine_type in enumerate(parcels3.swine_type_grp.unique()):
    plt.subplot(2, 2, i + 1)

    # Filter data for the specific swine type group
    group_900_950 = parcels3[
        (parcels3.animal_units >= 851)
        & (parcels3.animal_units <= 950)
        & (parcels3.swine_type_grp == swine_type)
    ].barn_area
    group_951_1000 = parcels3[
        (parcels3.animal_units >= 951)
        & (parcels3.animal_units <= 1000)
        & (parcels3.swine_type_grp == swine_type)
    ].barn_area
    group_1001_1050 = parcels3[
        (parcels3.animal_units >= 1001)
        & (parcels3.animal_units <= 1100)
        & (parcels3.swine_type_grp == swine_type)
    ].barn_area

    # Plot histograms with consistent bin width
    plt.hist(
        group_900_950,
        bins=bins,
        color="blue",
        alpha=0.3,
        label=f"851-950 Animal Units (N={len(group_900_950)})",
        density=True,
    )
    plt.hist(
        group_951_1000,
        bins=bins,
        color="green",
        alpha=0.5,
        label=f"951-1000 Animal Units (N={len(group_951_1000)})",
        density=True,
    )
    plt.hist(
        group_1001_1050,
        bins=bins,
        color="purple",
        alpha=0.3,
        label=f"1001-1100 Animal Units (N={len(group_1001_1050)})",
        density=True,
    )

    plt.title(f"{swine_type}")
    plt.xlabel("Barn Area (m^2)")
    plt.ylabel("Relative Frequency")
    # plt.yscale('log')  # Set y-axis to logarithmic scale
    # plt.xscale('log')  # Set x-axis to logarithmic scale
    plt.legend()

    plt.tight_layout()

    # Perform Levene's test for equality of variances
    levene_stat1, levene_p1 = levene(group_900_950, group_951_1000)
    levene_stat2, levene_p2 = levene(group_900_950, group_951_1000, group_1001_1050)

    # Perform Two-Sample Kolmogorov-Smirnov Test
    ks_stat1, ks_p1 = ks_2samp(group_900_950, group_951_1000)
    ks_stat2, ks_p2 = ks_2samp(group_951_1000, group_1001_1050)

    # Display test results
    print(
        f"{swine_type} - Levene's Test for Equality of Variances:\n"
        f"851-950 vs 951-1000 Animal Units: Statistic: {levene_stat1:.4f}, p-value: {levene_p1:.4f}\n"
        f"951-1000 vs 1001-1100 Animal Units: Statistic: {levene_stat2:.4f}, p-value: {levene_p2:.4f}"
    )

    print(
        f"{swine_type} - Two-Sample Kolmogorov-Smirnov Test:\n"
        f"851-950 vs 951-1000 Animal Units: Statistic: {ks_stat1:.4f}, p-value: {ks_p1:.4f}\n"
        f"951-1000 vs 1001-1100 Animal Units: Statistic: {ks_stat2:.4f}, p-value: {ks_p2:.4f}"
    )

# Save the plot
plt.savefig("output/plots/swine_type_barn_size_relative_histograms.png", dpi=300)
plt.show()

In [ ]:
# inspect top facilities for wean and grow under 1000 animal units
top_10 = (
    parcels3[
        parcels3.swine_type_grp.isin(["wean", "grow"])
        & (parcels3.animal_units <= 1000)
        & (parcels3.animal_units > 900)
    ]
    .sort_values(by="barn_area", ascending=False)
    .head(10)
)

In [ ]:
print(
    top_10[
        [
            "permit_id",
            "facilityna",
            "address",
            "city",
            "zip",
            "animal_units",
            "barn_area",
            "swine_type",
        ]
    ]
)

In [ ]:
fig = simple_map(
    "EPSG:4326",
    barns_within_parcels[barns_within_parcels.parcel_id.isin(top_10.parcel_id)],
    parcels[parcels.parcel_id.isin(top_10.parcel_id)],
    permits,
)

fig.save("output/plots/top_10_wean_grow_facilities.html")

In [ ]:
# Code Starts here! Helena

barn_parcels_exploded = parcels2[parcels2["animal_type"] == "swine"]
print(np.shape(barn_parcels_exploded))
barn_parcels_exploded = barn_parcels_exploded[
    barn_parcels_exploded["animal_units"] == barn_parcels_exploded["swine_animal_units"]
]
print(np.shape(barn_parcels_exploded))
# print(barn_parcels_exploded['swine_type'])
# print(barn_parcels_exploded.columns)
pd.set_option("display.max_rows", None)
print(barn_parcels_exploded["swine_type"].unique())

In [ ]:
print(barn_parcels_exploded["swine_type"])

In [ ]:
print(parcels2.columns)
print(type(barn_parcels_exploded["swine_type"].iloc[0]))

In [ ]:
# For Weights and Densities

FARROW_FACTOR = 1 / 4.32

weight_factors_path = "cafo_iowa/data/weight-factors.csv"
weights_and_densities = pd.read_csv(weight_factors_path, index_col=0)  #


# TK come back to this later
animal_density_factors = {
    "grow": FARROW_FACTOR,
    "wean": FARROW_FACTOR,
    "gestation": FARROW_FACTOR,
    "sow": FARROW_FACTOR,
    "gilt": FARROW_FACTOR,
    "nursery": FARROW_FACTOR,
}

DENSITY_K = 0.035
weights_and_densities["density"] = 1 / (
    DENSITY_K * (weights_and_densities["avg_max_weight"] ** 0.667)
)

# print(weights_and_densities)

# This is usually not commented out
weights_and_densities["density"] = weights_and_densities.apply(
    lambda x: (
        animal_density_factors[x.name]
        if x.name in animal_density_factors.keys()
        else x["density"]
    ),
    axis=1,
)
# print(weights_and_densities.loc['grow'])

# barn_parcels_exploded["density"] = barn_parcels_exploded["swine_type"].map(
#             lambda x: print(f"x is: {x}") or sum(weights_and_densities.loc[x.split('-')] for t in x.split('-'))
#         )
# print(barn_parcels_exploded['density'])

# state_weights, county_weights = get_weights_over_stages(permits)
# state_density_factor = (weights_and_densities["density"] * state_weights).sum()

# barn_parcels_exploded["density"] = (
#                 barn_parcels_exploded["density"]
#                 # .fillna(barn_parcels_exploded["County"].map(county_density_factors))
#                 .fillna(state_density_factor)
#             )


# barn_parcels_exploded["area_per_stage"] = (
#             barn_parcels_exploded["swine_animal_units"]
#             * (1/barn_parcels_exploded[f"density"])
#         )
# barn_parcels_exploded["stage_weight"] = barn_parcels_exploded.groupby(level=0)[
#             "area_per_stage"
#         ].transform(lambda x: x / x.sum() if any(x > 0) else 1 / len(x))

In [ ]:
def get_weights_over_stages(permits: pd.DataFrame):
    """Get permit counts shares over facility stages.

    Args:
        permits: a DataFrame of permits.

    Returns:
        A tuple of state and county weights.
    """
    # Get activity weights
    permits_exploded = pd.DataFrame(permits).explode(
        ["swine_animal_units", "swine_type"], ignore_index=False
    )

    activity_tots = permits_exploded.groupby("swine_type")["swine_animal_units"].sum()
    activity_weights = (activity_tots / activity_tots.sum()).rename("weights")

    activity_tots_cnty = permits_exploded.groupby(["city", "swine_type"])[
        "swine_animal_units"
    ].sum()
    activity_weights_cnty = (
        activity_tots_cnty / (activity_tots_cnty.groupby("city").sum() + 1e-6)
    ).rename("weights")

    return activity_weights, activity_weights_cnty

In [ ]:
# print(barn_parcels_exploded)
# print(weights_and_densities)


DEAD_SPACE_FACTOR = 0.1
IA_UTM_CRS = "EPSG:32615"


def get_density(barn_parcels_exploded: gpd.GeoDataFrame, area_crs: str = IA_UTM_CRS):
    # print(barn_parcels_exploded)
    exploded_barn_parcels = barn_parcels_exploded
    # print(exploded_barn_parcels['permit_id'])
    exploded_barn_parcels["swine_type"] = exploded_barn_parcels["swine_type"].astype(
        str
    )
    exploded_barn_parcels["swine_type"] = exploded_barn_parcels["swine_type"].str.split(
        "-"
    )
    exploded_barn_parcels = exploded_barn_parcels.explode(
        "swine_type", ignore_index=True
    )
    # print(exploded_barn_parcels['permit_id'])
    exploded_barn_parcels["density"] = exploded_barn_parcels["swine_type"].map(
        weights_and_densities["density"]
    )
    # print(exploded_barn_parcels['swine_type'])
    # print(exploded_barn_parcels['density'])
    return exploded_barn_parcels


def estimate_animal_counts(
    barn_parcels_exploded: gpd.GeoDataFrame, area_crs: str = IA_UTM_CRS
) -> pd.Series:
    """Estimate animal counts estimates from barn area for each permit in a trading group.

    Args:
        barn_parcels_exploded: barn parcels.
        area_crs: the CRS to use for area calculations.

    Returns:
        A series of estimated animal counts.
    """
    # print(barn_parcels_exploded['stage_weight'])
    # print(barn_parcels_exploded['geometry'])
    print(barn_parcels_exploded["density"])
    return (
        barn_parcels_exploded["stage_weight"]
        * barn_parcels_exploded["geometry"].to_crs(area_crs).area
        * barn_parcels_exploded["density"]
        * (1 - DEAD_SPACE_FACTOR)
    )

In [ ]:
area_crs = IA_UTM_CRS
# Calculating the Animal Estimation
barn_parcels_exploded = parcels2[parcels2["animal_type"] == "swine"]
barn_parcels_exploded = barn_parcels_exploded[
    barn_parcels_exploded["animal_units"] == barn_parcels_exploded["swine_animal_units"]
]
barn_parcels_exploded = get_density(barn_parcels_exploded)

# Area per stage and stage weight
swine_type_columns = {
    "grow": "swine_grow",
    "nursery": "swine_nurs",
    "sow": "swine_sow",
    "wean": "swine_wean",
    "gilt": "swine_gilt",
    "gestation": "swine_gest",
}


# Function to calculate 'area_per_stage' based on swine type
def calculate_area_per_stage(row, swine_type_columns):
    swine_type = row["swine_type"]
    # Get the correct column based on 'swine_type'
    swine_column = swine_type_columns.get(swine_type)

    # If swine_type is valid and the column exists, use it for calculation
    if swine_column:
        return row[swine_column] * (1 / row["density"])
    else:
        return None  # Return None if swine_type is not recognized


# Apply the function row-wise to calculate 'area_per_stage'
barn_parcels_exploded["area_per_stage"] = barn_parcels_exploded.apply(
    lambda row: calculate_area_per_stage(row, swine_type_columns), axis=1
)

# barn_parcels_exploded["area_per_stage"] = (
#             barn_parcels_exploded["swine_animal_units"]
#             * (1/barn_parcels_exploded[f"density"])
#         )

# print(barn_parcels_exploded['permit_id'])


# barn_parcels_exploded_new = barn_parcels_exploded.groupby('permit_id').agg(
#     swine_stages = ('swine_type', lambda x: list(x)),
#     total_area = ('area_per_stage', lambda x: list(x))
# ).reset_index()


# print(barn_parcels_exploded_new['total_area'])

barn_parcels_exploded["stage_weight"] = barn_parcels_exploded.groupby("permit_id")[
    "area_per_stage"
].transform(lambda x: x / x.sum() if any(x > 0) else 1 / len(x))

barn_parcels_exploded["estimated_animal_count"] = estimate_animal_counts(
    barn_parcels_exploded
)

barn_parcels_exploded_new = (
    barn_parcels_exploded.groupby("permit_id")
    .agg(
        total_animal_numbers=("estimated_animal_count", "sum"),
        swine_animal_units=("swine_animal_units", "first"),
        swine_stages=("swine_type", lambda x: list(x)),
        area=("barn_area", "first"),
        #  area = ('geometry', lambda x: x.to_crs(area_crs).area)
    )
    .reset_index()
)


# barn_parcels_exploded = barn_parcels_exploded.merge(barn_parcels_exploded_new, on='permit_id', how='left')


# print(barn_parcels_exploded['total_animal_numbers'])
# print(barn_parcels_exploded)

import matplotlib.pyplot as plt

# plt.scatter(barn_parcels_exploded['estimated_animal_count']*.03, barn_parcels_exploded['swine_animal_units'], c = 'orange')
# print(barn_parcels_exploded)
plt.scatter(
    barn_parcels_exploded_new["total_animal_numbers"] * 0.03,
    barn_parcels_exploded_new["swine_animal_units"],
    c="blue",
)
plt.title("Estimated v Permitted Swine Animal Units")
plt.xlabel("Estimated Swine Animal Units (numbers * .03)")
plt.ylabel("Permit Given Swine Animal Units")

In [ ]:
barn_parcels_exploded_new["area"]

In [ ]:
wean_df = barn_parcels_exploded_new[
    barn_parcels_exploded_new["swine_stages"].apply(lambda x: x == ["sow"])
]
# print(type(barn_parcels_exploded_new['swine_stages'][0]))
print(wean_df.columns)
print(wean_df)
wean_df.to_csv("~/Downloads/wean_df.csv")
plt.scatter(wean_df["swine_animal_units"], wean_df["area"], c="blue", alpha=0.5)
plt.xlabel("Permit Animal Units")
plt.ylabel("Estimated Animal Number")
plt.title("Estimated v Allowed Animal Units for Sow Hogs")
plt.yscale("log")
plt.xscale("log")
plt.axvline(500, color="red", linestyle="--")
plt.axvline(1000, color="red", linestyle="--")

# plt.xlim(0,4000)
# plt.ylim(0,4000)

In [ ]:
barn_parcels_exploded_new["swine_stages"]

In [ ]:
barn_parcels_exploded_new["difference"] = (
    barn_parcels_exploded_new["total_animal_numbers"]
    - barn_parcels_exploded_new["swine_animal_units"]
)

# Sort the DataFrame by the difference in descending order and select the top 10 rows
top_10_entries = barn_parcels_exploded_new.nlargest(10, "difference")

# Display the top 10 entries with the greatest difference
print(
    top_10_entries[
        [
            "permit_id",
            "swine_stages",
            "swine_animal_units",
            "total_animal_numbers",
            "difference",
        ]
    ]
)